In [11]:
!pip install statsmodels scikit-learn numpy pandas matplotlib seaborn

In [29]:
import numpy as np
import pandas as pd
import math
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt

np.random.seed(42)

In [48]:
# 1. Generate Synthetic Dataset

def generate_synthetic(n=1500):
    age = np.random.normal(45, 12, n).clip(18, 90)
    income = np.random.normal(60000, 15000, n).clip(1000, 250000)
    health = np.random.normal(50, 10, n)
    smoke = np.random.binomial(1, 0.3, n)
    exercise = np.random.normal(3, 1, n).clip(0, 10)

    logit_t = -3 + 0.015*age + 0.00003*income + 0.05*exercise + 0.04*health - 0.6*smoke
    p_t = 1 / (1 + np.exp(-logit_t))

    treatment = np.random.binomial(1, p_t)

    tau = 10
    outcome = 40 + tau*treatment + 0.1*age + 0.0004*income + 0.8*exercise + np.random.normal(0,5,n)

    df = pd.DataFrame({
        'id': np.arange(n),
        'age': age,
        'income': income,
        'health': health,
        'smoke': smoke,
        'exercise': exercise,
        'treatment': treatment,
        'outcome': outcome
    })
    return df
    

df = generate_synthetic(1500)
df.to_csv("synthetic_psm_data.csv", index=False)

print("synthetic_psm_data.csv saved successfully!")
df.head()

synthetic_psm_data.csv saved successfully!


,id,age,income,health,smoke,exercise,treatment,outcome
0,0,46.387161,56600.066944,48.094748,0,2.523024,1,77.937162
1,1,44.642802,69737.215816,52.913745,0,3.686847,1,82.198320
2,2,48.859014,53201.107111,53.270752,1,3.695746,0,73.233723
3,3,46.398097,85776.594234,51.546485,0,3.191442,1,92.839654
4,4,49.556396,40654.858531,54.945225,0,3.006925,1,71.167001


In [31]:
# 2. Preprocessing

covariates = ['age','income','health','smoke','exercise']

imp = SimpleImputer(strategy='median')
df[covariates] = imp.fit_transform(df[covariates])

scaler = StandardScaler()
df[covariates] = scaler.fit_transform(df[covariates])

In [32]:
# 3. Propensity Score Model

X = df[covariates].values
y = df['treatment'].values

ps_model = LogisticRegression(max_iter=2000)
ps_model.fit(X, y)

df['ps'] = ps_model.predict_proba(X)[:,1]
df['ps'] = df['ps'].clip(1e-6, 1-1e-6)
df['ps_logit'] = np.log(df['ps'] / (1 - df['ps']))

In [33]:
# 4. Matching (1:1 NN, no replacement, caliper)

caliper = 0.2 * df['ps_logit'].std()

treated = df[df['treatment']==1].reset_index(drop=True)
control = df[df['treatment']==0].reset_index(drop=True)

nn = NearestNeighbors(n_neighbors=1).fit(control[['ps_logit']])
distances, indices = nn.kneighbors(treated[['ps_logit']])

matched_pairs = []
used_ctrl = set()

for i, (d, idx) in enumerate(zip(distances.ravel(), indices.ravel())):
    if abs(treated.loc[i,'ps_logit'] - control.loc[idx,'ps_logit']) <= caliper:
        if idx not in used_ctrl:
            matched_pairs.append((treated.loc[i,'id'], control.loc[idx,'id']))
            used_ctrl.add(idx)

In [34]:
# 5. Build matched dataset (flat dict rows)

matched_rows = []

for gid, (tid, cid) in enumerate(matched_pairs):
    rt = df[df['id']==tid].iloc[0].to_dict()
    rc = df[df['id']==cid].iloc[0].to_dict()

    rt['matched_group'] = gid
    rc['matched_group'] = gid

    matched_rows.append(rt)
    matched_rows.append(rc)

matched_df = pd.DataFrame(matched_rows).reset_index(drop=True)
matched_df.to_csv("matched_dataset.csv", index=False)


In [35]:
# 6. Compute Standardized Mean Differences

def smd(a, b):
    m1, m0 = a.mean(), b.mean()
    s1, s0 = a.std(ddof=1), b.std(ddof=1)
    pooled = np.sqrt((s1**2 + s0**2)/2)
    return 0 if pooled==0 else (m1 - m0)/pooled

balance = []
for v in covariates:
    before = smd(df[df['treatment']==1][v],
                 df[df['treatment']==0][v])
    after = smd(matched_df[matched_df['treatment']==1][v],
                matched_df[matched_df['treatment']==0][v])

    balance.append({
        'covariate': v,
        'SMD_before': before,
        'SMD_after': after,
        'treated_mean_before': df[df['treatment']==1][v].mean(),
        'control_mean_before': df[df['treatment']==0][v].mean(),
        'treated_mean_after': matched_df[matched_df['treatment']==1][v].mean(),
        'control_mean_after': matched_df[matched_df['treatment']==0][v].mean()
    })

balance_df = pd.DataFrame(balance)
balance_df.to_csv("balance_table.csv", index=False)

In [36]:
# 7. Love Plot (auto detect SMD columns)

cols = [c.lower() for c in balance_df.columns]
smd_before_col = balance_df.columns[cols.index('smd_before')]
smd_after_col = balance_df.columns[cols.index('smd_after')]

plt.figure(figsize=(7, max(4, len(balance_df)*0.5)))
y = np.arange(len(balance_df))
plt.scatter(balance_df[smd_before_col], y, label='Before', marker='o')
plt.scatter(balance_df[smd_after_col], y+0.15, label='After', marker='s')
plt.axvline(0.1, color='gray', linestyle='--')
plt.axvline(-0.1, color='gray', linestyle='--')
plt.yticks(y, balance_df['covariate'])
plt.xlabel("Standardized Mean Difference")
plt.title("Love Plot")
plt.legend()
plt.tight_layout()
plt.savefig("love_plot.png", dpi=150)
plt.close()


In [37]:
# 8. ATT estimation (paired)

pairs = matched_df.pivot_table(index='matched_group',
                               columns='treatment',
                               values='outcome')
pairs = pairs.dropna()
pairs = pairs.rename(columns={0:'control', 1:'treated'})
pairs['diff'] = pairs['treated'] - pairs['control']

ATT = pairs['diff'].mean()
SE = pairs['diff'].std(ddof=1) / np.sqrt(len(pairs))

# bootstrap CI
def bootstrap_ci(arr, n_boot=1000, alpha=0.05):
    rng = np.random.default_rng(42)
    boot = [rng.choice(arr, len(arr), replace=True).mean() for _ in range(n_boot)]
    return np.percentile(boot, alpha/2*100), np.percentile(boot, (1-alpha/2)*100)

ci_lo, ci_hi = bootstrap_ci(pairs['diff'].values)

In [38]:
# 9. Save Report

with open("psm_report.txt", "w") as f:
    f.write("Propensity Score Matching Report\n")
    f.write("================================\n\n")
    f.write(f"Total N: {len(df)}\n")
    f.write(f"Treated: {df['treatment'].sum()}\n")
    f.write(f"Controls: {len(df) - df['treatment'].sum()}\n")
    f.write(f"Matched pairs: {matched_df['matched_group'].nunique()}\n")
    f.write(f"Caliper: {caliper:.4f}\n\n")

    f.write("ATT Estimation:\n")
    f.write(f"ATT = {ATT:.4f}\n")
    f.write(f"SE  = {SE:.4f}\n")
    f.write(f"95% CI = [{ci_lo:.4f}, {ci_hi:.4f}]\n\n")

    f.write("Balance Table:\n")
    f.write(balance_df.to_string(index=False))

In [57]:
print("=== PSM COMPLETED SUCCESSFULLY ===")
print(f"Matched pairs: {matched_df['matched_group'].nunique()}")
print(f"ATT: {ATT:.4f}")
print(f"95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")
print("Files saved: synthetic_psm_data.csv, matched_dataset.csv, balance_table.csv, love_plot.png, psm_report.txt")

=== PSM COMPLETED SUCCESSFULLY ===
Matched pairs: 248
ATT: 9.3045
95% CI: [8.2306, 10.3256]
Files saved: synthetic_psm_data.csv, matched_dataset.csv, balance_table.csv, love_plot.png, psm_report.txt
